# Customer Churn Prediction — Kaggle, without deployment

Adapted from [satyamshivam13/Customer_Churn_Prediction](https://github.com/satyamshivam13/Customer_Churn_Prediction) (MIT).
Dataset: Bank Customer Churn (`Churn_Modelling.csv`), supplied in the uploaded project.

**Setup:** Import this notebook into Kaggle. Attach `Customer_Churn_Prediction-main.zip`
as a private dataset, or attach its extracted files. Keep accelerator **None (CPU)**.
Enable Internet if the dependency cell needs to install a missing package. Run the cells in order.

This version trains Logistic Regression, Random Forest and XGBoost. It uses a 60/20/20
stratified train/validation/test split, training-only imputation/encoding/scaling,
and one shared feature transformation for training and new-customer predictions.
Validation ROC-AUC selects the model; only the selected model is evaluated on test.
Results will differ from the original repository's 80/20 split.
No FastAPI, Docker, SQL, or hosting is required. No uploaded `.joblib` files are loaded.

## 1. Check dependencies

In [ ]:
import importlib.util
import subprocess
import sys

# Reuse Kaggle's installed libraries; install only missing packages.
packages = {'pandas':'pandas', 'numpy':'numpy', 'sklearn':'scikit-learn',
            'matplotlib':'matplotlib', 'seaborn':'seaborn',
            'joblib':'joblib', 'xgboost':'xgboost==2.1.4'}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Dependencies ready. Python:', sys.version.split()[0])

## 2. Load the dataset automatically

In [ ]:
from pathlib import Path
import os, zipfile, hashlib, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn, xgboost
from IPython.display import display

# Kaggle defaults. Environment overrides are only for running this notebook locally.
INPUT_ROOT = Path(os.environ.get('CHURN_INPUT_ROOT', '/kaggle/input'))
OUTPUT_DIR = Path(os.environ.get('CHURN_OUTPUT_DIR', '/kaggle/working/churn_outputs'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
sns.set_theme(style='whitegrid')

csv_files = sorted(INPUT_ROOT.rglob('Churn_Modelling.csv'))
if len(csv_files) > 1:
    raise ValueError(f'Multiple datasets found. Set INPUT_ROOT to the intended dataset folder: {csv_files}')
if csv_files:
    source = csv_files[0]
    raw_bytes = source.read_bytes()
    df = pd.read_csv(source)
    source_description = str(source)
else:
    candidates = []
    for archive in sorted(INPUT_ROOT.rglob('*.zip')):
        if not zipfile.is_zipfile(archive):
            continue
        with zipfile.ZipFile(archive) as z:
            for member in z.namelist():
                if Path(member).name == 'Churn_Modelling.csv':
                    candidates.append((archive, member))
    if len(candidates) != 1:
        raise FileNotFoundError('Attach one project ZIP or dataset containing Churn_Modelling.csv. '
                                f'Found {len(candidates)} matching CSVs inside ZIP files.')
    archive, member = candidates[0]
    from io import BytesIO
    with zipfile.ZipFile(archive) as z:
        raw_bytes = z.read(member)
    df = pd.read_csv(BytesIO(raw_bytes))
    source_description = f'{archive} :: {member}'

data_sha256 = hashlib.sha256(raw_bytes).hexdigest()
print('Loaded:', source_description)
print('Dataset shape:', df.shape)
print('Outputs:', OUTPUT_DIR)
display(df.head())

## 3. Validate and clean data

In [ ]:
BASE_NUMERIC = ['CreditScore','Age','Tenure','Balance','NumOfProducts',
                'HasCrCard','IsActiveMember','EstimatedSalary']
BASE_CATEGORICAL = ['Geography','Gender']
FEATURES = BASE_NUMERIC + BASE_CATEGORICAL
TARGET = 'Exited'
required = FEATURES + [TARGET, 'CustomerId']
absent = sorted(set(required) - set(df.columns))
if absent:
    raise ValueError(f'Missing required columns: {absent}')

quality = {'initial_rows':len(df), 'exact_duplicates':int(df.duplicated().sum()),
           'missing_by_column':df[required].isna().sum().astype(int).to_dict()}
df = df.drop_duplicates().copy()
if df['CustomerId'].isna().any() or df['CustomerId'].duplicated().any():
    raise ValueError('CustomerId must be present and unique to avoid customer overlap across splits.')
for col in BASE_NUMERIC + [TARGET]:
    df[col] = pd.to_numeric(df[col], errors='raise')
if df[TARGET].isna().any() or not df[TARGET].isin([0,1]).all():
    raise ValueError('Exited must contain only 0 or 1, without missing values.')
if np.isinf(df[BASE_NUMERIC].to_numpy(dtype=float)).any():
    raise ValueError('Numeric features contain infinite values.')
for col in BASE_CATEGORICAL:
    df[col] = df[col].map(lambda v: v.strip() if isinstance(v, str) else v)
    df[col] = df[col].replace('', np.nan)
quality['cleaned_rows'] = len(df)
quality['churn_rate'] = float(df[TARGET].mean())
(OUTPUT_DIR / 'data_quality.json').write_text(json.dumps(quality, indent=2))
print('Rows after removing exact duplicates:', len(df))
print(f'Churn rate: {df[TARGET].mean():.2%}')
print('Missing features will be imputed using training data only.')
display(df[required].isna().sum().to_frame('missing'))

## 4. Split before fitting any preprocessing

In [ ]:
from sklearn.model_selection import train_test_split

train_df, holdout_df = train_test_split(df, test_size=0.4, stratify=df[TARGET], random_state=SEED)
val_df, test_df = train_test_split(holdout_df, test_size=0.5,
                                  stratify=holdout_df[TARGET], random_state=SEED)
X_train, y_train = train_df[FEATURES], train_df[TARGET].astype(int)
X_val, y_val = val_df[FEATURES], val_df[TARGET].astype(int)
X_test, y_test = test_df[FEATURES], test_df[TARGET].astype(int)
train_ids, val_ids, test_ids = [set(part['CustomerId']) for part in [train_df,val_df,test_df]]
assert train_ids.isdisjoint(val_ids) and train_ids.isdisjoint(test_ids) and val_ids.isdisjoint(test_ids)
split_summary = pd.DataFrame([
    {'split':name, 'rows':len(part), 'churn_rate':part[TARGET].mean()}
    for name,part in [('train',train_df),('validation',val_df),('test',test_df)]
])
display(split_summary)
split_summary.to_csv(OUTPUT_DIR / 'split_summary.csv', index=False)
pd.concat([part[['CustomerId']].assign(split=name) for name,part in
           [('train',train_df),('validation',val_df),('test',test_df)]],
          ignore_index=True).to_csv(OUTPUT_DIR / 'split_assignments.csv', index=False)

## 5. Explore the training data

In [ ]:
# Use training data for exploration to keep the test set untouched.
fig, axes = plt.subplots(1, 3, figsize=(16,4))
sns.countplot(data=train_df, x=TARGET, ax=axes[0])
axes[0].set_title('Training: stayed (0) vs exited (1)')
sns.boxplot(data=train_df, x=TARGET, y='Age', ax=axes[1])
axes[1].set_title('Training: age by churn')
train_df.groupby('Geography')[TARGET].mean().plot.bar(ax=axes[2])
axes[2].set_title('Training: churn rate by geography')
axes[2].set_ylabel('Churn rate')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_eda.png', dpi=150)
plt.show()

## 6. Shared feature engineering and preprocessing

The generated helper module makes the saved model pipeline reusable. Its feature formulas
match the original training code (with safe group boundaries). Both training and inference
use this same transformer. One-hot encoding replaces label encoding for nominal categories.
Training data alone supplies imputation values, encoder categories and scaling statistics.

In [ ]:
helper_source = r"""
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class ChurnFeatureBuilder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['TenureGroup'] = pd.cut(X['Tenure'], bins=[0,3,6,9,11],
            labels=['0-3 years','3-6 years','6-9 years','9+ years'], include_lowest=True).astype(object)
        X['AgeGroup'] = pd.cut(X['Age'], bins=[0,30,45,60,100],
            labels=['Young','Middle','Senior','Elderly'], include_lowest=True).astype(object)
        X['BalanceToSalaryRatio'] = X['Balance'] / (X['EstimatedSalary'] + 1)
        X['HighProductRisk'] = np.where(X['NumOfProducts'].notna(),
            (X['NumOfProducts'] >= 3).astype(float), np.nan)
        X['EngagementScore'] = X['IsActiveMember'] + X['NumOfProducts'] / 4
        return X
"""
helper_path = OUTPUT_DIR / 'churn_preprocessing.py'
helper_path.write_text(helper_source)
if str(OUTPUT_DIR) not in sys.path:
    sys.path.insert(0, str(OUTPUT_DIR))
import importlib
importlib.invalidate_caches()
from churn_preprocessing import ChurnFeatureBuilder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

NUMERIC = BASE_NUMERIC + ['BalanceToSalaryRatio','HighProductRisk','EngagementScore']
CATEGORICAL = BASE_CATEGORICAL + ['TenureGroup','AgeGroup']
def build_preprocessor():
    columns = ColumnTransformer([
        ('numeric', Pipeline([('impute',SimpleImputer(strategy='median')),
                              ('scale',StandardScaler())]), NUMERIC),
        ('categorical', Pipeline([('impute',SimpleImputer(strategy='most_frequent')),
                                  ('encode',OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
         CATEGORICAL)
    ])
    return Pipeline([('features',ChurnFeatureBuilder()),('columns',columns)])
print('Shared preprocessing ready.')

## 7. Train three models and compare on validation data

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)

estimators = {
    'logistic_regression':LogisticRegression(max_iter=1000, random_state=SEED),
    'random_forest':RandomForestClassifier(n_estimators=100, max_depth=10,
        min_samples_split=20, min_samples_leaf=10, n_jobs=2, random_state=SEED),
    'xgboost':XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
        tree_method='hist', n_jobs=2, random_state=SEED)
}
def evaluate(model, X, y):
    probability = model.predict_proba(X)[:,1]
    prediction = (probability >= 0.5).astype(int)
    return {'accuracy':accuracy_score(y,prediction),
            'precision':precision_score(y,prediction,zero_division=0),
            'recall':recall_score(y,prediction,zero_division=0),
            'f1':f1_score(y,prediction,zero_division=0),
            'roc_auc':roc_auc_score(y,probability)}

trained_models, validation_rows = {}, []
for name, estimator in estimators.items():
    print('Training:', name)
    pipeline = Pipeline([('preprocessing',build_preprocessor()),('model',estimator)])
    pipeline.fit(X_train,y_train)
    trained_models[name] = pipeline
    validation_rows.append({'model':name, **evaluate(pipeline,X_val,y_val)})
    joblib.dump(pipeline, OUTPUT_DIR / f'{name}_pipeline.joblib')

validation_metrics = pd.DataFrame(validation_rows).sort_values('roc_auc', ascending=False)
display(validation_metrics)
validation_metrics.to_csv(OUTPUT_DIR / 'validation_model_comparison.csv',index=False)
best_name = validation_metrics.iloc[0]['model']
best_pipeline = trained_models[best_name]
print('Selected using validation ROC-AUC:',best_name)

## 8. Evaluate the selected model on the untouched test set

In [ ]:
test_metrics = {'model':best_name, **evaluate(best_pipeline,X_test,y_test)}
display(pd.DataFrame([test_metrics]))
pd.DataFrame([test_metrics]).to_csv(OUTPUT_DIR / 'selected_model_test_metrics.csv',index=False)
test_probability = best_pipeline.predict_proba(X_test)[:,1]
test_prediction = (test_probability >= 0.5).astype(int)
print(classification_report(y_test,test_prediction,
                            target_names=['Stayed','Exited'],zero_division=0))
test_output = pd.DataFrame({'CustomerId':test_df['CustomerId'].to_numpy(),
                           'actual_churn':y_test.to_numpy(),
                           'predicted_churn':test_prediction,
                           'churn_probability':test_probability})
test_output.to_csv(OUTPUT_DIR / 'test_predictions.csv',index=False)
fig,axes = plt.subplots(1,2,figsize=(12,4))
ConfusionMatrixDisplay.from_predictions(y_test,test_prediction,
    display_labels=['Stayed','Exited'],ax=axes[0],colorbar=False)
axes[0].set_title(f'{best_name}: test confusion matrix')
RocCurveDisplay.from_predictions(y_test,test_probability,ax=axes[1])
axes[1].plot([0,1],[0,1],'--',color='grey')
axes[1].set_title(f'{best_name}: test ROC curve')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'selected_model_test_plots.png',dpi=150)
plt.show()

# Save this same evaluated pipeline, without refitting on the validation/test data.
joblib.dump(best_pipeline,OUTPUT_DIR / 'best_churn_pipeline.joblib')
metadata = {'selected_model':best_name,'selection_metric':'validation ROC-AUC',
            'decision_threshold':0.5,'seed':SEED,'split_fraction':{'train':0.6,'validation':0.2,'test':0.2},
            'features':FEATURES,'source':source_description,'data_sha256':data_sha256,
            'test_metrics':test_metrics,
            'versions':{'python':sys.version.split()[0],'pandas':pd.__version__,
                        'numpy':np.__version__,'scikit-learn':sklearn.__version__,
                        'xgboost':xgboost.__version__,'joblib':joblib.__version__}}
(OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(metadata,indent=2))

## 9. Inspect tree-model feature importance
This is built-in tree importance, not a causal explanation. The chart uses Random Forest
even if another model wins validation selection. Encoded categories have separate entries.

In [ ]:
rf_pipeline = trained_models['random_forest']
names = rf_pipeline.named_steps['preprocessing'].named_steps['columns'].get_feature_names_out()
importance = pd.DataFrame({'feature':names,
    'importance':rf_pipeline.named_steps['model'].feature_importances_}).sort_values('importance',ascending=False)
importance.to_csv(OUTPUT_DIR / 'random_forest_feature_importance.csv',index=False)
display(importance.head(15))
top = importance.head(15).sort_values('importance')
plt.figure(figsize=(9,6))
plt.barh(top['feature'],top['importance'])
plt.title('Random Forest: feature importance')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'random_forest_feature_importance.png',dpi=150)
plt.show()

## 10. Predict for new customers — no API required

In [ ]:
# Edit these examples to test other customers.
new_customers = pd.DataFrame([
    {'CreditScore':750,'Geography':'France','Gender':'Female','Age':35,
     'Tenure':8,'Balance':120000.0,'NumOfProducts':2,'HasCrCard':1,
     'IsActiveMember':1,'EstimatedSalary':80000.0},
    {'CreditScore':450,'Geography':'Germany','Gender':'Male','Age':55,
     'Tenure':2,'Balance':150000.0,'NumOfProducts':3,'HasCrCard':0,
     'IsActiveMember':0,'EstimatedSalary':40000.0}
])
saved_pipeline = joblib.load(OUTPUT_DIR / 'best_churn_pipeline.joblib')
probabilities = saved_pipeline.predict_proba(new_customers[FEATURES])[:,1]
new_results = new_customers.copy()
new_results['churn_probability'] = probabilities
new_results['predicted_churn'] = np.where(probabilities >= 0.5,'Yes','No')
display(new_results)
new_results.to_csv(OUTPUT_DIR / 'new_customer_predictions.csv',index=False)
print('These are example predictions, not validated customer outcomes.')

## 11. Package your outputs
The ZIP includes model pipelines, the shared preprocessing module, metrics, predictions,
plots, split assignments and environment versions. Keep `churn_preprocessing.py` alongside
the saved pipelines and import it before loading them in a new Python session.
Use trusted model files only. Kaggle Save Version / Run All preserves notebook outputs.

Limitations: this is a bank benchmark; no temporal observation dates are available, so
the split is stratified random rather than chronological. The decision threshold is fixed
at 0.5. Class imbalance is reported, but SMOTE and threshold tuning are not implemented.
No deployment or automated monitoring service is included.

In [ ]:
import shutil
archive_path = shutil.make_archive(str(OUTPUT_DIR.parent / 'customer_churn_outputs'),
                                   'zip',root_dir=OUTPUT_DIR)
print('Download:',archive_path)
print('Saved files:')
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(' -',path.name)